# 🌀 SIFE-LDM V4.0 — Physics-Informed Multimodal Diffusion

**Structured Intelligence Field Latent Diffusion Model**

This notebook implements the architectural reconstruction of SIFE-LDM v4.0. It utilizes **Dual-Modality Diffusion**:
1. **Vision:** Continuous Polar Diffusion (Gaussian on Amplitude, Wrapped Gaussian on Phase).
2. **Text:** Masked Discrete Diffusion (Iterative phase prediction for semantic tokens).

**Dataset:** `lambdalabs/pokemon-blip-captions` (Multimodal Image-Text pairs).

## 1️⃣ Setup Environment

In [ ]:
!pip install -q flax optax datasets transformers
import jax
import jax.numpy as jnp
import numpy as np
print(f'JAX Version: {jax.__version__}')
print(f'Devices: {jax.devices()}')

## 2️⃣ Clone SIFE-LDM v4.0 Repository
Cloning directly from GitHub to ensure latest architectural fixes.

In [ ]:
!rm -rf SIFE-LDM-
!git clone https://github.com/vicekha/SIFE-LDM-.git
%cd SIFE-LDM-
import sys
sys.path.append('.')

from sife.model import SIFELDM, SIFELDMConfig, get_loss
from sife.field import SIFEConfig
from sife.diffusion import GaussianDiffusion, MaskedDiffusion
from sife.optim import andi
from sife.tokenizer import Vocabulary, SIFETokenizer
from train import create_train_state, train_step_vision_jit, train_step_text_jit

## 3️⃣ Prepare Pokemon Dataset (Multimodal)

In [ ]:
from datasets import load_dataset
from PIL import Image

print("Loading Pokemon-BLIP dataset...")
ds = load_dataset('lambdalabs/pokemon-blip-captions', split='train')

IMAGE_SIZE = 128 # Using 128 for Colab stability

def collate_fn(batch):
    imgs = []
    texts = []
    for item in batch:
        imgs.append(np.array(item['image'].resize((IMAGE_SIZE, IMAGE_SIZE)).convert('RGB')))
        texts.append(item['text'])
    
    imgs = np.array(imgs).astype(np.float32) / 255.0
    return {'image': imgs, 'text': texts}

# Setup Tokenizer
vocab = Vocabulary()
vocab.build_from_texts([item['text'] for item in ds])
tokenizer = SIFETokenizer(vocab, max_len=64)

print(f"Vocab size: {len(vocab)}")

## 4️⃣ Initialize v4.0 Model

In [ ]:
config = SIFELDMConfig(
    vocab_size=len(vocab),
    max_seq_len=64,
    embed_dim=256,
    num_layers=6,
    num_heads=8,
    mlp_dim=512,
    patch_size=8,
    image_size=IMAGE_SIZE,
    physics_config=SIFEConfig(gamma=0.2, alpha=0.1) # Physics-informed stability
)

rng = jax.random.PRNGKey(0)
state = create_train_state(config, rng)
print("Model Initialized with Hamiltonian Physics Engine.")

## 5️⃣ Training Loop (Interleaved Vision/Text)

In [ ]:
BATCH_SIZE = 8
STEPS = 1000

def get_batch(start_idx):
    batch_raw = ds[start_idx : start_idx + BATCH_SIZE]
    processed = collate_fn([{'image': img, 'text': txt} for img, txt in zip(batch_raw['image'], batch_raw['text'])])
    tokenized = tokenizer(processed['text'])
    return {
        'image': processed['image'],
        'tokens': tokenized['input_ids'],
        'mask': tokenized['mask']
    }

for step in range(STEPS):
    start_idx = (step * BATCH_SIZE) % (len(ds) - BATCH_SIZE)
    batch = get_batch(start_idx)
    
    rng, v_rng, t_rng = jax.random.split(rng, 3)
    
    # Interleaved step
    state, v_loss = train_step_vision_jit(state, {'image': batch['image']}, v_rng, config=config)
    state, t_loss = train_step_text_jit(state, {'tokens': batch['tokens'], 'mask': batch['mask']}, t_rng, config=config)
    
    if step % 50 == 0:
        print(f"Step {step} | Vision Loss: {v_loss:.4f} | Text Loss: {t_loss:.4f}")

## 6️⃣ Generation

In [ ]:
from inference import generate_vision, generate_text
import matplotlib.pyplot as plt

# Generate Image
rng, gen_rng = jax.random.split(rng)
model = SIFELDM(config)
gen_imgs = generate_vision(model, state.params, config, gen_rng, batch_size=1)

plt.imshow(gen_imgs[0])
plt.axis('off')
plt.show()

# Generate Text
prompt = "A fire type pokemon with"
start_ids = tokenizer([prompt], padding=False)['input_ids'][0]
gen_tokens = generate_text(model, state.params, config, gen_rng, start_tokens=start_ids[None, :], max_new_tokens=20)
print("Generated Pokemon Description:", vocab.decode(gen_tokens[0].tolist()))